<a href="https://colab.research.google.com/github/angelolucero-cloud/GO-Grupo-2/blob/main/PARTE_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# CARGA DE RESULTADOS DE PARTE B
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files
import zipfile

uploaded = files.upload()

nombre_zip = next(iter(uploaded))

carpeta_b = Path("/content/salidas_parte_c")
carpeta_b.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(nombre_zip, "r") as zip_ref:
    zip_ref.extractall(carpeta_b)

print("Archivos disponibles:")

for archivo in carpeta_b.iterdir():
    print("-", archivo.name)

Saving salidas_parte_c.zip to salidas_parte_c.zip
Archivos disponibles:
- sigma_error_SKU_horizonte.csv
- pronosticos_13_semanas.csv


**Resultado esperado.**

El archivo comprimido proveniente de la Parte B debe contener dos resultados principales:

- `pronosticos_13_semanas.csv`: contiene el pronóstico puntual de cada SKU para las próximas 13 semanas.
- `sigma_error_SKU_horizonte.csv`: contiene la desviación estándar del error histórico para cada combinación SKU-horizonte.

Estos archivos constituyen los parámetros de pronóstico que serán incorporados posteriormente a la tabla final de traspaso.

### 1. Resultados heredados de Pronóstico

La Parte B seleccionó el método **Bottom-Up** para generar el pronóstico final, debido a que presentó el mejor desempeño global en la comparación jerárquica.

Para la Parte C se necesitan dos resultados provenientes de dicho análisis.

El primero corresponde al pronóstico puntual:

$$
\hat{d}_{i,t}
$$

donde \(i\) representa el SKU y \(t\) corresponde a cada una de las 13 semanas futuras.

El segundo corresponde a:

$$
\hat{\sigma}_{i,h}
$$

que representa la desviación estándar del error histórico del pronóstico para cada SKU \(i\) y horizonte \(h\).

Mientras el pronóstico indica cuántas cajas se espera vender, la desviación estándar entrega una medida de la incertidumbre asociada a dicha estimación.

In [7]:
# ============================================================
# CARGAR PRONÓSTICOS Y SIGMA
# ============================================================

pronosticos_b = pd.read_csv(
    carpeta_b / "pronosticos_13_semanas.csv",
    parse_dates=["Fecha"]
)

sigma_error = pd.read_csv(
    carpeta_b / "sigma_error_SKU_horizonte.csv"
)

print("PRONÓSTICOS RECIBIDOS DESDE PARTE B")
display(pronosticos_b.head(15).round(2))

print("\nSIGMA DEL ERROR POR SKU Y HORIZONTE")
display(sigma_error.head(15).round(2))

print("\nCOMPROBACIÓN")

print("Pronósticos:", len(pronosticos_b))

print(
    "SKU:",
    pronosticos_b["SKU"].nunique()
)

print(
    "Horizontes:",
    pronosticos_b["Horizonte"].nunique()
)

print(
    "Combinaciones sigma:",
    len(sigma_error)
)

print(
    "Sigmas faltantes:",
    sigma_error["sigma_error_cajas"].isna().sum()
)

PRONÓSTICOS RECIBIDOS DESDE PARTE B


,Fecha,Horizonte,SKU,Pronostico
0,2026-09-28,1,CIT-L05,1392.00
1,2026-10-05,2,CIT-L05,1753.00
2,2026-10-12,3,CIT-L05,2080.00
3,2026-10-19,4,CIT-L05,1740.00
4,2026-10-26,5,CIT-L05,2042.00
5,2026-11-02,6,CIT-L05,2097.00
6,2026-11-09,7,CIT-L05,2126.00
7,2026-11-16,8,CIT-L05,3321.00
8,2026-11-23,9,CIT-L05,2326.00
9,2026-11-30,10,CIT-L05,2579.00



SIGMA DEL ERROR POR SKU Y HORIZONTE


,SKU,Horizonte,sigma_error_cajas,n_errores
0,CIT-L05,1,433.15,8
1,CIT-L05,2,261.13,8
2,CIT-L05,3,659.05,8
3,CIT-L05,4,10663.34,8
4,CIT-L05,5,912.03,8
5,CIT-L05,6,419.15,8
6,CIT-L05,7,228.61,8
7,CIT-L05,8,645.23,8
8,CIT-L05,9,10587.86,8
9,CIT-L05,10,922.92,8



COMPROBACIÓN
Pronósticos: 117
SKU: 9
Horizontes: 13
Combinaciones sigma: 117
Sigmas faltantes: 0


La Parte C tiene como objetivo reunir los resultados obtenidos en Calidad y Pronóstico y convertirlos en parámetros que posteriormente serán utilizados en la planificación de producción e inventario.
Desde la Parte A se utilizarán la tasa de merma efectiva por línea y el sobrellenado medio por formato. Desde la Parte B se utilizarán los pronósticos finales de demanda para las próximas 13 semanas y la incertidumbre asociada a dichos pronósticos.
El resultado final será una tabla de traspaso con una fila para cada combinación SKU–semana del horizonte, de manera que la Entrega 2 pueda utilizar directamente estos parámetros.

**Verificación de los resultados recibidos.**

La base debe contener 9 SKU y 13 semanas de pronóstico. Por lo tanto, se esperan:

$$
9 \times 13 = 117
$$

combinaciones SKU-semana.

De forma equivalente, deben existir 117 valores de desviación estándar del error, uno para cada combinación SKU-horizonte.

También se verifica que no existan valores faltantes de $$
\hat{\sigma}_{i,h}
$$, ya que este parámetro será utilizado posteriormente para definir el stock de seguridad en la planificación.

Si los resultados muestran 117 pronósticos, 9 SKU, 13 horizontes, 117 combinaciones de sigma y 0 valores faltantes, el traspaso desde la Parte B se considera completo.

### 2. Parámetros heredados de Calidad

Desde la Parte A se incorporan dos resultados principales: la tasa de merma efectiva por línea y el sobrellenado medio por formato.

La tasa de merma se obtuvo a partir de la carta **p**:

$$
\rho_{L1}=0.0192
$$

$$
\rho_{L2}=0.0311
$$

Esto significa que aproximadamente el 1,92 % de la producción de L1 y el 3,11 % de la producción de L2 no termina como producto vendible.

Por esta razón, si se necesitan \(x\) cajas vendibles, la producción bruta deberá ajustarse mediante:

$$
x^{bruto}=\frac{x}{1-\rho_\ell}
$$

También se incorporan los valores de sobrellenado obtenidos en Calidad. Este exceso representa bebida adicional contenida en cada botella y, por lo tanto, aumenta el consumo efectivo de mezcla por caja.

In [8]:
# ============================================================
# PARÁMETROS HEREDADOS DE CALIDAD
# ============================================================

parametros_calidad = pd.DataFrame({
    "linea_envasado": ["L1", "L2", "L2"],
    "formato_L": [0.5, 1.5, 3.0],
    "rho_merma": [0.0192, 0.0311, 0.0311],
    "sobrellenado_mL_botella": [5.720, 4.633, 8.429],
    "sobrellenado_L_caja": [0.06864, 0.027798, 0.033716]
})

parametros_calidad["factor_produccion_bruta"] = (
    1 / (1 - parametros_calidad["rho_merma"])
)

print("PARÁMETROS HEREDADOS DE CALIDAD")
display(parametros_calidad.round(5))

PARÁMETROS HEREDADOS DE CALIDAD


,linea_envasado,formato_L,rho_merma,sobrellenado_mL_botella,sobrellenado_L_caja,factor_produccion_bruta
0,L1,0.5,0.0192,5.720,0.06864,1.01958
1,L2,1.5,0.0311,4.633,0.02780,1.03210
2,L2,3.0,0.0311,8.429,0.03372,1.03210


**Interpretación**

L2 presenta una merma mayor que L1, por lo que deberá producir una cantidad bruta ligeramente superior para obtener una misma cantidad de cajas vendibles.

El factor de producción de L1 es aproximadamente:

$$
\frac{1}{1-0.0192}=1.0196
$$

mientras que para L2 es:

$$
\frac{1}{1-0.0311}=1.0321
$$

Esto significa que, por cada 1.000 cajas vendibles requeridas, L1 deberá producir aproximadamente 1.020 cajas y L2 cerca de 1.032 cajas.

En cuanto al sobrellenado, los valores obtenidos fueron:

- Formato 0,5 L: **0,06864 L adicionales por caja**.
- Formato 1,5 L: **0,02780 L adicionales por caja**.
- Formato 3,0 L: **0,03372 L adicionales por caja**.

Estos valores deberán considerarse posteriormente al calcular el consumo efectivo de mezcla.

### 3. Integración de Pronóstico, incertidumbre y Calidad

El siguiente paso consiste en reunir los parámetros provenientes de las Partes A y B en una única estructura.

Para cada combinación SKU-semana se incorporará:

- el pronóstico puntual de demanda;
- la desviación estándar histórica del error de pronóstico;
- la línea de envasado;
- el formato;
- la tasa de merma de la línea;
- el sobrellenado correspondiente al formato;
- los litros nominales y efectivos consumidos por caja.

De esta forma, cada SKU queda asociado tanto a su demanda futura esperada como a los parámetros operacionales que afectarán la planificación de producción.

In [9]:
# ============================================================
# UNIÓN DE PRONÓSTICO + SIGMA
# ============================================================

tabla_traspaso = pronosticos_b[
    ["Fecha", "Horizonte", "SKU", "Pronostico"]
].copy()

tabla_traspaso = tabla_traspaso.rename(
    columns={"Pronostico": "pronostico_cajas"}
)

tabla_traspaso = tabla_traspaso.merge(
    sigma_error,
    on=["SKU", "Horizonte"],
    how="left",
    validate="one_to_one"
)

print("PRONÓSTICO + INCERTIDUMBRE")
display(tabla_traspaso.head(15).round(2))

print("\nFilas:", len(tabla_traspaso))
print(
    "Sigmas faltantes:",
    tabla_traspaso["sigma_error_cajas"].isna().sum()
)

PRONÓSTICO + INCERTIDUMBRE


,Fecha,Horizonte,SKU,pronostico_cajas,sigma_error_cajas,n_errores
0,2026-09-28,1,CIT-L05,1392.00,433.15,8
1,2026-10-05,2,CIT-L05,1753.00,261.13,8
2,2026-10-12,3,CIT-L05,2080.00,659.05,8
3,2026-10-19,4,CIT-L05,1740.00,10663.34,8
4,2026-10-26,5,CIT-L05,2042.00,912.03,8
5,2026-11-02,6,CIT-L05,2097.00,419.15,8
6,2026-11-09,7,CIT-L05,2126.00,228.61,8
7,2026-11-16,8,CIT-L05,3321.00,645.23,8
8,2026-11-23,9,CIT-L05,2326.00,10587.86,8
9,2026-11-30,10,CIT-L05,2579.00,922.92,8



Filas: 117
Sigmas faltantes: 0


**Interpretación.**

Cada pronóstico puntual queda ahora acompañado por una medida de su incertidumbre.

Por ejemplo, si un SKU presenta un pronóstico de 1.000 cajas y una desviación estándar del error elevada, significa que históricamente los pronósticos realizados a ese horizonte han presentado una mayor variabilidad.

El pronóstico y la desviación estándar cumplen funciones diferentes: el primero representa la demanda esperada, mientras que la segunda será utilizada posteriormente para determinar el stock de seguridad.

### 4. Incorporación de información productiva

Los parámetros de Calidad no se asignan directamente por SKU.

La tasa de merma depende de la **línea de envasado**, mientras que el sobrellenado depende del **formato de la botella**.

Por esta razón, se utiliza el maestro de SKU para relacionar cada producto con su línea, formato, cantidad de botellas por caja y litros nominales por caja.

In [12]:
# ============================================================
# CARGAR MAESTRO DE SKU
# ============================================================

from pathlib import Path
from google.colab import files

rutas_posibles = [
    Path("/content/GO-Grupo-2/datos/maestro_sku.csv"),
    Path("/content/maestro_sku.csv")
]

ruta_maestro = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if ruta_maestro is None:

    print("Selecciona el archivo maestro_sku.csv")
    uploaded_maestro = files.upload()

    nombre_maestro = next(iter(uploaded_maestro))
    ruta_maestro = Path(nombre_maestro)

maestro = pd.read_csv(ruta_maestro)

print("MAESTRO DE SKU")
display(maestro.head())

print("\nSKU disponibles:", maestro["sku"].nunique())

Selecciona el archivo maestro_sku.csv


Saving maestro_sku.csv to maestro_sku.csv
MAESTRO DE SKU


,sku,familia,nombre_familia,sabor,formato_L,botellas_por_caja,litros_por_caja,linea_envasado,tasa_envasado_cajas_h,costo_variable_caja_clp,precio_lista_caja_clp,costo_inventario_caja_semana_clp,penalidad_atraso_caja_semana_clp
0,NEC-D05,NEC,Nectares con pulpa,Durazno,0.5,12,6.0,L1,180,3840,6620,95,1400
1,NEC-D15,NEC,Nectares con pulpa,Durazno,1.5,6,9.0,L2,210,5760,9930,95,1400
2,NEC-D30,NEC,Nectares con pulpa,Durazno,3.0,4,12.0,L2,130,7680,13240,95,1400
3,CIT-L05,CIT,Citricos gasificados,Limon,0.5,12,6.0,L1,180,2820,5320,95,1400
4,CIT-L15,CIT,Citricos gasificados,Limon,1.5,6,9.0,L2,210,4230,7980,95,1400



SKU disponibles: 9


In [13]:
# ============================================================
# AGREGAR INFORMACIÓN PRODUCTIVA
# ============================================================

tabla_traspaso = tabla_traspaso.merge(
    maestro[
        [
            "sku",
            "linea_envasado",
            "formato_L",
            "botellas_por_caja",
            "litros_por_caja"
        ]
    ],
    left_on="SKU",
    right_on="sku",
    how="left",
    validate="many_to_one"
)

tabla_traspaso = tabla_traspaso.drop(columns="sku")

print("PRONÓSTICO + INFORMACIÓN PRODUCTIVA")
display(tabla_traspaso.head(15).round(2))

PRONÓSTICO + INFORMACIÓN PRODUCTIVA


,Fecha,Horizonte,SKU,pronostico_cajas,sigma_error_cajas,n_errores,linea_envasado,formato_L,botellas_por_caja,litros_por_caja
0,2026-09-28,1,CIT-L05,1392.00,433.15,8,L1,0.5,12,6.0
1,2026-10-05,2,CIT-L05,1753.00,261.13,8,L1,0.5,12,6.0
2,2026-10-12,3,CIT-L05,2080.00,659.05,8,L1,0.5,12,6.0
3,2026-10-19,4,CIT-L05,1740.00,10663.34,8,L1,0.5,12,6.0
4,2026-10-26,5,CIT-L05,2042.00,912.03,8,L1,0.5,12,6.0
5,2026-11-02,6,CIT-L05,2097.00,419.15,8,L1,0.5,12,6.0
6,2026-11-09,7,CIT-L05,2126.00,228.61,8,L1,0.5,12,6.0
7,2026-11-16,8,CIT-L05,3321.00,645.23,8,L1,0.5,12,6.0
8,2026-11-23,9,CIT-L05,2326.00,10587.86,8,L1,0.5,12,6.0
9,2026-11-30,10,CIT-L05,2579.00,922.92,8,L1,0.5,12,6.0


**Interpretación.**

Cada pronóstico de demanda se encuentra ahora asociado a las características físicas del producto.

Esto permite saber, por ejemplo, si un SKU se produce en L1 o L2 y si corresponde al formato de 0,5 L, 1,5 L o 3,0 L.

Esta relación es necesaria porque los parámetros de merma y sobrellenado son diferentes según la línea y el formato.


### 5. Incorporación de merma y sobrellenado

A continuación se incorporan los resultados obtenidos en la Parte A.

La tasa de merma afecta la cantidad que debe producirse para obtener una determinada cantidad de producto vendible.

El sobrellenado afecta una dimensión diferente: aumenta la cantidad de litros de mezcla utilizados por cada caja producida.

Por lo tanto:

- la **merma** corrige el rendimiento de la producción;
- el **sobrellenado** corrige el consumo de mezcla.

In [14]:
# ============================================================
# INCORPORAR PARÁMETROS DE CALIDAD
# ============================================================

tabla_traspaso = tabla_traspaso.merge(
    parametros_calidad,
    on=["linea_envasado", "formato_L"],
    how="left",
    validate="many_to_one"
)

print("TABLA CON PARÁMETROS DE CALIDAD")
display(tabla_traspaso.head(15).round(4))

TABLA CON PARÁMETROS DE CALIDAD


,Fecha,Horizonte,SKU,pronostico_cajas,sigma_error_cajas,n_errores,linea_envasado,formato_L,botellas_por_caja,litros_por_caja,rho_merma,sobrellenado_mL_botella,sobrellenado_L_caja,factor_produccion_bruta
0,2026-09-28,1,CIT-L05,1392.0000,433.1481,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
1,2026-10-05,2,CIT-L05,1753.0000,261.1300,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
2,2026-10-12,3,CIT-L05,2080.0000,659.0491,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
3,2026-10-19,4,CIT-L05,1740.0000,10663.3371,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
4,2026-10-26,5,CIT-L05,2042.0000,912.0318,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
5,2026-11-02,6,CIT-L05,2097.0000,419.1512,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
6,2026-11-09,7,CIT-L05,2126.0000,228.6088,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
7,2026-11-16,8,CIT-L05,3321.0000,645.2326,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
8,2026-11-23,9,CIT-L05,2326.0000,10587.8606,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196
9,2026-11-30,10,CIT-L05,2579.0000,922.9188,8,L1,0.5,12,6.0,0.0192,5.720,0.0686,1.0196


### 6. Consumo efectivo de mezcla

El contenido nominal de una caja no representa exactamente la cantidad de mezcla que utiliza el proceso, debido al sobrellenado observado.

Por ello, el consumo efectivo por caja se calcula como:

$$
L_i^{efectivo}
=
L_i^{nominal}
+
L_i^{sobrellenado}
$$

Este parámetro será utilizado en la Entrega 2 al formular la restricción de capacidad de mezcla.

In [15]:
# ============================================================
# CONSUMO EFECTIVO DE MEZCLA
# ============================================================

tabla_traspaso["litros_efectivos_caja"] = (
    tabla_traspaso["litros_por_caja"]
    + tabla_traspaso["sobrellenado_L_caja"]
)

# Comprobación del sobrellenado
tabla_traspaso["sobrellenado_calculado_L_caja"] = (
    tabla_traspaso["sobrellenado_mL_botella"]
    * tabla_traspaso["botellas_por_caja"]
    / 1000
)

assert np.allclose(
    tabla_traspaso["sobrellenado_L_caja"],
    tabla_traspaso["sobrellenado_calculado_L_caja"]
)

print("CONSUMO EFECTIVO POR CAJA")

display(
    tabla_traspaso[
        [
            "SKU",
            "formato_L",
            "litros_por_caja",
            "sobrellenado_L_caja",
            "litros_efectivos_caja"
        ]
    ]
    .drop_duplicates()
    .round(5)
)

CONSUMO EFECTIVO POR CAJA


,SKU,formato_L,litros_por_caja,sobrellenado_L_caja,litros_efectivos_caja
0,CIT-L05,0.5,6.0,0.06864,6.06864
13,CIT-L15,1.5,9.0,0.02780,9.02780
26,CIT-L30,3.0,12.0,0.03372,12.03372
39,FUN-B05,0.5,6.0,0.06864,6.06864
52,FUN-B15,1.5,9.0,0.02780,9.02780
65,FUN-B30,3.0,12.0,0.03372,12.03372
78,NEC-D05,0.5,6.0,0.06864,6.06864
91,NEC-D15,1.5,9.0,0.02780,9.02780
104,NEC-D30,3.0,12.0,0.03372,12.03372


### 7. Tabla final de traspaso hacia la Entrega 2

Finalmente se consolidan los cuatro parámetros exigidos para el traspaso:

1. tasa de merma efectiva por línea;
2. sobrellenado medio por formato;
3. pronóstico puntual de demanda para las próximas 13 semanas;
4. desviación estándar del error de pronóstico por SKU y horizonte.

Cada fila representa una combinación SKU-semana.

La tasa de merma se mantiene como parámetro y no se transforma todavía en una cantidad fija de producción, ya que en la Entrega 2 la producción será una variable de decisión del modelo de planificación.

In [16]:
# ============================================================
# TABLA FINAL DE TRASPASO
# ============================================================

tabla_final_c = tabla_traspaso[
    [
        "Fecha",
        "Horizonte",
        "SKU",
        "linea_envasado",
        "formato_L",

        "pronostico_cajas",
        "sigma_error_cajas",
        "n_errores",

        "rho_merma",
        "factor_produccion_bruta",

        "sobrellenado_mL_botella",
        "sobrellenado_L_caja",

        "litros_por_caja",
        "litros_efectivos_caja"
    ]
].copy()

tabla_final_c = (
    tabla_final_c
    .sort_values(["SKU", "Horizonte"])
    .reset_index(drop=True)
)

print("TABLA FINAL DE TRASPASO")
display(tabla_final_c.round(4))

TABLA FINAL DE TRASPASO


,Fecha,Horizonte,SKU,linea_envasado,formato_L,pronostico_cajas,sigma_error_cajas,n_errores,rho_merma,factor_produccion_bruta,sobrellenado_mL_botella,sobrellenado_L_caja,litros_por_caja,litros_efectivos_caja
0,2026-09-28,1,CIT-L05,L1,0.5,1392.0000,433.1481,8,0.0192,1.0196,5.720,0.0686,6.0,6.0686
1,2026-10-05,2,CIT-L05,L1,0.5,1753.0000,261.1300,8,0.0192,1.0196,5.720,0.0686,6.0,6.0686
2,2026-10-12,3,CIT-L05,L1,0.5,2080.0000,659.0491,8,0.0192,1.0196,5.720,0.0686,6.0,6.0686
3,2026-10-19,4,CIT-L05,L1,0.5,1740.0000,10663.3371,8,0.0192,1.0196,5.720,0.0686,6.0,6.0686
4,2026-10-26,5,CIT-L05,L1,0.5,2042.0000,912.0318,8,0.0192,1.0196,5.720,0.0686,6.0,6.0686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,2026-11-23,9,NEC-D30,L2,3.0,567.6649,115.3474,8,0.0311,1.0321,8.429,0.0337,12.0,12.0337
113,2026-11-30,10,NEC-D30,L2,3.0,427.3304,147.3635,8,0.0311,1.0321,8.429,0.0337,12.0,12.0337
114,2026-12-07,11,NEC-D30,L2,3.0,510.0017,175.6274,8,0.0311,1.0321,8.429,0.0337,12.0,12.0337
115,2026-12-14,12,NEC-D30,L2,3.0,545.9988,157.7178,8,0.0311,1.0321,8.429,0.0337,12.0,12.0337


### 8. Verificación de consistencia

Antes de utilizar la tabla en la Entrega 2 se verifica que la integración entre Calidad y Pronóstico se haya realizado correctamente.

Deben existir 9 SKU y 13 horizontes, equivalentes a:

$$
9 \times 13 = 117
$$

filas en la tabla final.

También se verifica que no existan valores faltantes en los parámetros esenciales de planificación.

In [17]:
# ============================================================
# COMPROBACIONES FINALES
# ============================================================

columnas_clave = [
    "pronostico_cajas",
    "sigma_error_cajas",
    "rho_merma",
    "sobrellenado_L_caja",
    "litros_efectivos_caja"
]

print("COMPROBACIONES FINALES")

print("Filas esperadas:", 117)
print("Filas obtenidas:", len(tabla_final_c))

print(
    "SKU:",
    tabla_final_c["SKU"].nunique()
)

print(
    "Horizontes:",
    tabla_final_c["Horizonte"].nunique()
)

print(
    "Valores faltantes en parámetros clave:",
    tabla_final_c[columnas_clave].isna().sum().sum()
)

print(
    "Sigma faltantes:",
    tabla_final_c["sigma_error_cajas"].isna().sum()
)

assert len(tabla_final_c) == 117
assert tabla_final_c["SKU"].nunique() == 9
assert tabla_final_c["Horizonte"].nunique() == 13
assert tabla_final_c[columnas_clave].isna().sum().sum() == 0

print("\nIntegración completada correctamente.")

COMPROBACIONES FINALES
Filas esperadas: 117
Filas obtenidas: 117
SKU: 9
Horizontes: 13
Valores faltantes en parámetros clave: 0
Sigma faltantes: 0

Integración completada correctamente.


In [18]:
# ============================================================
# EXPORTAR TABLA FINAL DE PARTE C
# ============================================================

from google.colab import files

tabla_final_c.to_csv(
    "/content/tabla_traspaso_entrega2.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

files.download("/content/tabla_traspaso_entrega2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusión de la Parte C

La Parte C permitió integrar los resultados obtenidos en Calidad y Pronóstico en una única tabla de parámetros para la planificación de producción.

Desde Calidad se incorporaron las tasas de merma de 1,92 % para L1 y 3,11 % para L2, además del sobrellenado medio correspondiente a cada formato. Estos parámetros permitirán ajustar tanto el rendimiento de la producción como el consumo efectivo de mezcla.

Desde Pronóstico se incorporaron los pronósticos finales de demanda de los nueve SKU para las próximas 13 semanas y la desviación estándar histórica del error para cada combinación SKU-horizonte.

Como resultado, se obtuvo una tabla de 117 combinaciones SKU-semana que contiene los parámetros necesarios para la Entrega 2: demanda esperada, incertidumbre del pronóstico, merma y sobrellenado.

De esta forma, la Entrega 1 deja definidos los parámetros que serán utilizados posteriormente en las restricciones de producción, capacidad de mezcla, inventario y stock de seguridad.